In [23]:
import re
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [38]:
def load_pdfs(pdf_dir):
    documents = []
    pdf_dir = Path(pdf_dir)

    for pdf in pdf_dir.glob("**/*.pdf"):
        loader = PyPDFLoader(str(pdf))
        docs = loader.load()

        for d in docs:
            d.metadata["source"] = pdf.name

        documents.extend(docs)

    return documents

In [39]:
docs = load_pdfs("../data")   # <-- your PDF folder
print("Pages loaded:", len(docs))

Pages loaded: 57


In [40]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)
print("Chunks:", len(chunks))

Chunks: 191


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

C:\Users\Sanika\AppData\Local\Temp\ipykernel_37964\2539825802.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 814.52it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


: 

: 

In [37]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 772.08it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
llm = Ollama(
    model="llama3",
    temperature=0
)

In [35]:
def retrieve_context(query, vectorstore, llm, k=10):
    # 1️⃣ Dense retrieval (FAISS)
    dense_docs = vectorstore.similarity_search(query, k=k)

    # 2️⃣ Lexical retrieval (BM25)
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    top_bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    bm25_docs = [chunks[i] for i in top_bm25_indices]

    # 3️⃣ Merge + deduplicate
    docs = {d.page_content: d for d in dense_docs + bm25_docs}.values()

    context = "\n\n".join(d.page_content for d in docs)
    return context

In [33]:
def generate_answer(query, context, llm):
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present, say "Not found in provided documents".

Context:
{context}

Question:
{query}

Answer:
"""
    return llm.invoke(prompt).strip()

In [34]:
def rewrite_query(query, llm):
    prompt = f"""
Rewrite the question as a factual academic search query.

Question:
{query}

Rewritten query:
"""
    return llm.invoke(prompt).strip()

In [31]:
def extract_claims(answer, llm):
    prompt = f"""
Split the following answer into atomic factual claims.
Each claim should be independently verifiable.

Answer:
{answer}

Return each claim on a new line.
"""
    response = llm.invoke(prompt)

    claims = [
        c.strip("- ").strip()
        for c in response.split("\n")
        if c.strip()
    ]
    return claims

In [30]:
from collections import Counter

def verify_claim_once(claim, context, llm):
    prompt = f"""
Context:
{context}

Claim:
{claim}

Decide if the claim is EXPLICITLY supported by the context.
SUPPORTED = clearly stated or directly paraphrased.
NOT SUPPORTED = inferred, assumed, or missing.

Answer ONLY:
SUPPORTED or NOT SUPPORTED
"""
    return llm.invoke(prompt).content.strip()

def verify_claim(claim, context, llm, runs=3):
    votes = []
    for _ in range(runs):
        votes.append(verify_claim_once(claim, context, llm))

    final_decision = Counter(votes).most_common(1)[0][0]
    return final_decision, votes


In [29]:
def aggregate_claim_results(claim_results):
    """
    claim_results: list of tuples
    Each tuple can be:
      (claim, final_decision)
    where final_decision is 'SUPPORTED' or 'NOT SUPPORTED'
    """

    if not claim_results:
        return "NOT HALLUCINATION", 0.0

    total = len(claim_results)
    unsupported = sum(
        1 for _, decision in claim_results 
        if decision.strip().upper() == "NOT SUPPORTED"
    )

    hallucination_rate = unsupported / total

    if hallucination_rate == 0:
        verdict = "NOT HALLUCINATION"
    elif hallucination_rate == 1:
        verdict = "HALLUCINATION"
    else:
        verdict = "PARTIAL HALLUCINATION"

    return verdict, hallucination_rate


In [28]:
def extract_claims(answer, llm):
    prompt = f"""
Extract all factual, verifiable claims from the answer below.
Each claim must be ONE complete statement.
Return as a numbered list.

Answer:
{answer}
"""
    response = llm.invoke(prompt).content
    return [c.strip() for c in response.split("\n") if c.strip()]



In [26]:
def claimwise_rag_pipeline(query, vectorstore, llm):
    # ✅ pass llm here
    context = retrieve_context(query, vectorstore, llm)

    answer = generate_answer(query, context, llm)

    if answer.lower().startswith("not found"):
        return {
            "question": query,
            "answer": answer,
            "hallucination_verdict": "NOT ANSWERED",
            "hallucination_rate": 0.0,
            "claims": []
        }

    claims = extract_claims(answer, llm)

    claim_results = []
    for claim in claims:
        verdict = verify_claim(claim, context, llm)
        claim_results.append((claim, verdict))

    final_verdict, hallucination_rate = aggregate_claim_results(claim_results)

    return {
        "question": query,
        "answer": answer,
        "claims": claim_results,
        "hallucination_verdict": final_verdict,
        "hallucination_rate": hallucination_rate
    }

In [27]:
def is_trivial_answer(answer):
    return len(answer.split()) <= 2  # "Yes", "No", "Maybe"

In [25]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [24]:
from rank_bm25 import BM25Okapi

# Build BM25 corpus once
bm25_corpus = [doc.page_content for doc in chunks]
tokenized_corpus = [doc.split() for doc in bm25_corpus]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
query = "What recent techniques have been developed to infer whether a large language model is telling the truth?"

result = claimwise_rag_pipeline(query, vectorstore, llm)

print("Question:", result["question"])
print("\nAnswer:\n", result["answer"])

print("\nClaim Analysis:")
for c in result["claims"]:
    print(c)

print("\nHallucination Verdict:", result["hallucination_verdict"])
print("Hallucination Rate:", result["hallucination_rate"])

Question: What recent techniques have been developed to infer whether a large language model is telling the truth?

Answer:
 Not found. The provided context does not mention any specific techniques for inferring whether a large language model is telling the truth. However, it discusses various methods for detecting hallucinations in LLMs, including probing for truthfulness and causal intervention experiments.

Claim Analysis:

Hallucination Verdict: NOT ANSWERED
Hallucination Rate: 0.0


: 

In [ ]:
result = claimwise_rag_pipeline(query, vectorstore, llm)

print("=== RETRIEVED CONTEXT ===")
print(result["context"][:3000])
print("=========================")

=== RETRIEVED CONTEXT ===
model generations [Luo et al., 2023], while others have proposed frameworks that design prompts
with diagnostic questions to probe the extent of hallucinations [Manakul et al., 2023, Muhammed
et al., 2025, Yang et al., 2025]. Complementing these approaches, uncertainty estimation methods
have been introduced to quantify the inherent ambiguity in model predictions – a factor often observed
in hallucinated outputs [Varshney et al., 2023, Manakul et al., 2023, Rateike et al., 2023].
It remains unclear whether any one of these evaluation methods is sufficient to detect all classes of
hallucinations. Hallucination patterns can manifest in different ways, making it unlikely that a single
metric can detect hallucinations across diverse datasets and models. Additionally, given the rapid

datasets, such as FactCHD, where low ROUGE-L scores hinder the collection of sufficient prompts
that do not generate hallucinations. Alternative strategies, such as leveraging ‘LLM-as

: 

: 

: 

: 

: 

: 